# Lab 05: The Multiagent Pattern for Evidence and Custody Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danjethh/sectratek-AI-Lab/blob/main/lab05_multiagent_pattern/03_lab_notebook.ipynb)

**Course:** AI for Cybersecurity and Digital Forensics  
**Instructor:** Daniel Kwaku Ntiamoah Addai, Mobile Threat Research Lead, SectraTEK Lab  
**Runs in:** Google Colab or any hosted Jupyter. Nothing is installed on your own computer.

---

The first four labs each used **one** agent. Some investigations are better split across **several**
specialists who check one another. That is the **Multiagent Pattern**: each agent has a narrow role,
and its output becomes context for the next.

Three agents review one case: an **InvestigationAgent** builds the technical timeline, an
**EvidenceVerificationAgent** tests its claims against the records, and a **CustodyAuditAgent** checks
how the evidence was handled and writes the final, confidence-bounded conclusion. You run them one at a
time (Part A), then let a **Crew** orchestrate the same team in dependency order (Part B).

## Part 0: One-Time Setup

Run these cells in order. They install one library, connect you to a **free hosted model** (Groq),
write the case files, and load the multiagent framework. No local install is required.

### Step 0.1: Install the client library

**Purpose:** install the `openai` client, the one small library that lets the notebook talk to the model (Groq uses the same interface).  
**What you'll see:** a short pip log. Warnings are normal.

In [ ]:
# Purpose: install the one library this notebook needs, inside the hosted session only.
%pip install -q openai

### Step 0.2: Connect to a free hosted model (Groq)

**Purpose:** get a free **API key** (a private password that lets the notebook use the model) and build a client.

1. Open **https://console.groq.com/keys**, sign in, and create a key.
2. Run the cell and paste the key when asked. It is hidden and is **not saved** into the notebook.

**What you'll see:** `Model OK: ...` once the connection works.

In [ ]:
# Purpose: collect the Groq key privately and build an OpenAI-compatible client pointed at Groq.
import os        # os: lets the notebook read saved settings, such as where your key is kept
import getpass   # getpass: lets you type your secret key without it appearing on screen
from openai import OpenAI  # OpenAI: the client that carries your messages to the model (Groq speaks the same language)

if not os.environ.get('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass.getpass('Paste your Groq API key (hidden): ').strip()

BASE_URL = 'https://api.groq.com/openai/v1'
# Free models that work well for this lab, best first. Groq retires models from time to time, so
# this cell uses the first one your key can reach and prints which. Edit the list to try another.
# Qwen comes first here: this lab's agent writes <tool_call> tags as plain text, which Qwen follows;
# gpt-oss tends to plan the call silently and return an empty reply.
PREFERRED = ['qwen/qwen3.8-27b', 'openai/gpt-oss-120b']

# max_retries=8: the free tier allows about 8,000 tokens a minute. When a call goes over, Groq says
# 'try again in a few seconds' and the client waits and retries instead of stopping with an error.
client = OpenAI(base_url=BASE_URL, api_key=os.environ['GROQ_API_KEY'], max_retries=8)

# Every model call in this notebook goes through this small wrapper. It does two things:
# 1. Groq counts each request's prompt PLUS its reply limit against that budget, so each call gets a
#    reply limit: 1,500 tokens, or 900 for Qwen, whose free tier allows 1,000 reply tokens a
#    minute (any call can still set its own max_tokens).
# 2. gpt-oss sometimes tries a built-in function call instead of writing text, which Groq rejects
#    ('model called a tool'). The wrapper then asks again with a plain-text reminder.
_create = client.chat.completions.create
def _create_within_limits(*args, **kw):
    kw.setdefault('max_tokens', 900 if 'qwen' in str(kw.get('model')) else 1500)
    for attempt in range(3):
        try:
            return _create(*args, **kw)
        except Exception as err:
            if 'called a tool' not in str(err) or attempt == 2:
                raise
            kw['messages'] = list(kw.get('messages', [])) + [
                {'role': 'user', 'content': 'Reply in plain text only; do not call functions.'}]
client.chat.completions.create = _create_within_limits

available = [m.id for m in client.models.list().data]
MODEL = next((m for m in PREFERRED if m in available), None)
if MODEL is None:
    print('None of PREFERRED is on your key right now. Models your key can use:')
    for m in available:
        print('  -', m)
    raise SystemExit('Put one of these chat models first in PREFERRED and re-run this cell.')

probe = client.chat.completions.create(
    model=MODEL, messages=[{'role': 'user', 'content': 'Reply with one word: ready'}])
print('Model OK:', MODEL, '->', (probe.choices[0].message.content or '').strip())

### Step 0.3: Write the synthetic case files

**Purpose:** create a local `data/` folder with the six artifacts for case `MA-2026-022`. These are
synthetic and identical to the repository's copies.  
**What you'll see:** the files written. Existing files are not overwritten.

In [ ]:
# Purpose: create the case files on disk so the notebook has everything it needs, with no download.
from pathlib import Path  # Path: a readable way to refer to files and folders by name

ARTIFACTS = {'artifact_manifest.json': '{\n  "case_id": "MA-2026-022",\n  "lab": "Lab 05 - Multiagent Pattern",\n  "device": {\n    "platform": "Android",\n    "model": "Motorola Edge 2024",\n    "os_version": "Android 14"\n  },\n  "acquisition": {\n    "type": "full_file_system",\n    "tool": "Cellebrite UFED 7.70",\n    "acquired_at_utc": "2026-03-03T12:04:55Z",\n    "original_image_sha256": "2ef6f6d371459ce23d3953196b03f979f8d2d440d4e8f1bf13d1712d7bf7f3a1"\n  },\n  "analysis_timezone": "America/New_York",\n  "incident_window_utc": {\n    "start": "2026-03-01T18:35:00Z",\n    "end": "2026-03-01T19:10:00Z"\n  },\n  "note": "Synthetic classroom dataset for multiagent-pattern demo"\n}\n', 'device_state.csv': 'timestamp_utc,event,notes\n2026-03-01T18:35:02Z,device_locked,"Phone left unattended during vaccination event"\n2026-03-01T19:10:08Z,device_unlocked,"Phone recovered by staff member"\n', 'file_events.csv': 'timestamp_utc,path,event,sha256\n2026-03-01T18:44:11Z,/storage/emulated/0/Pictures/patients_contacts.png,created,91c01aa8ea5bd91ab2d4f0a2c2f9f8189517588fb9f76ff4e777e7345b4870ac\n', 'messaging_events.csv': 'timestamp_utc,app,event,target,details\n2026-03-01T18:45:02Z,SecureChat,attach_attempt,external_contact_7,"patients_contacts.png selected as outbound attachment"\n', 'network_events.csv': 'timestamp_utc,app,event,bytes_out,notes\n2026-03-01T18:45:07Z,SecureChat,upload_started,48211,"Outbound upload session opened"\n', 'chain_of_custody.csv': 'timestamp_utc,handler,action,notes\n2026-03-03T12:04:55Z,Investigator_A,acquired_image,"UFED acquisition to encrypted evidence drive"\n2026-03-03T12:41:08Z,Investigator_A,computed_hash,"SHA-256 verified against acquisition log"\n2026-03-04T09:15:30Z,Analyst_B,created_working_copy,"Exported device state, file, messaging, and network artifacts"\n2026-03-05T11:02:44Z,Analyst_C,review_started,"Review began, but no transfer entry from Analyst_B was documented"\n'}

data_dir = Path('data'); data_dir.mkdir(exist_ok=True)
for name, text in ARTIFACTS.items():
    fp = data_dir / name
    if not fp.exists():
        fp.write_text(text)
    print(('wrote ' if fp.stat().st_size else 'exists ') + str(fp))

### Step 0.4: Load the multiagent framework

**Purpose:** load the tool wrapper, the `ReactAgent` each team member runs on, and the `Agent` and
`Crew` classes that connect them. This is a self-contained copy adapted
from The Neural Maze's agentic-patterns-course (MIT License).  
**What you'll see:** nothing printed; it just defines the classes used below.

In [ ]:
# Purpose: the multiagent framework (@tool, ReactAgent, Agent, Crew). Self-contained copy for this notebook.
# Adapted from The Neural Maze agentic-patterns-course (MIT License), (c) 2024 The Neural Maze.

import json
import re
from dataclasses import dataclass
from typing import Callable


# ---------------------------------------------------------------- tag extraction
@dataclass
class TagContentResult:
    content: list
    found: bool


def extract_tag_content(text: str, tag: str) -> TagContentResult:
    """Extract everything enclosed by <tag>...</tag> (used for <thought>, <tool_call>, <response>)."""
    if tag == "tool_call":
        text = re.sub(r"(</tool_call>\s*){2,}$", "</tool_call>", text.strip())
    pattern = rf"<{tag}>(.*?)</{tag}>"
    matches = re.findall(pattern, text, re.DOTALL)
    return TagContentResult(content=[m.strip() for m in matches], found=bool(matches))


# ---------------------------------------------------------------- chat history
def build_prompt_structure(prompt: str, role: str, tag: str = "") -> dict:
    if tag:
        prompt = f"<{tag}>{prompt}</{tag}>"
    return {"role": role, "content": prompt}


def update_chat_history(history: list, msg: str, role: str):
    history.append(build_prompt_structure(prompt=msg, role=role))


class ChatHistory(list):
    def __init__(self, messages=None, total_length=-1):
        super().__init__(messages or [])
        self.total_length = total_length

    def append(self, msg):
        if len(self) == self.total_length:
            self.pop(0)
        super().append(msg)


def completions_create(client, messages, model) -> str:
    response = client.chat.completions.create(messages=list(messages), model=model)
    return str(response.choices[0].message.content)


# ---------------------------------------------------------------- tool wrapper
def get_fn_signature(fn: Callable) -> dict:
    sig = {"name": fn.__name__, "description": fn.__doc__, "parameters": {"properties": {}}}
    schema = {k: {"type": v.__name__} for k, v in fn.__annotations__.items() if k != "return"}
    sig["parameters"]["properties"] = schema
    return sig


def validate_arguments(tool_call: dict, tool_signature: dict) -> dict:
    properties = tool_signature["parameters"]["properties"]
    type_mapping = {"int": int, "str": str, "bool": bool, "float": float}
    validated = {}
    for name, value in tool_call["arguments"].items():
        schema = properties.get(name)
        if schema is None:
            continue
        converter = type_mapping.get(schema.get("type"))
        if converter is None:
            validated[name] = value
            continue
        if not isinstance(value, converter):
            value = converter(value)
        validated[name] = value
    tool_call["arguments"] = validated
    return tool_call


class Tool:
    def __init__(self, name, fn, fn_signature):
        self.name = name
        self.fn = fn
        self.fn_signature = fn_signature

    def __str__(self):
        return self.fn_signature

    def run(self, **kwargs):
        return self.fn(**kwargs)


def tool(fn: Callable) -> Tool:
    """Decorator: turn a plain function into a Tool (its name, docstring, and typed parameters)."""
    sig = get_fn_signature(fn)
    return Tool(name=sig["name"], fn=fn, fn_signature=json.dumps(sig))


# ---------------------------------------------------------------- the ReAct agent
REACT_SYSTEM_PROMPT = """
## Role
You are a function-calling AI model that reasons and acts in a loop until you can deliver a final response to the user.

## Loop Structure
Each iteration of your loop consists of three steps:

1. **Thought** - Reason about the current situation and decide what to do next. Wrap in <thought></thought> tags.
2. **Action** - Call one tool by emitting a <tool_call> JSON object (see format below).
3. **Observation** - The tool result will be returned wrapped in <observation></observation> tags. Use it to inform your next thought.

Repeat this loop until you have enough information to give a final answer, then wrap it in <response></response> tags.

## Tool Call Format
<tool_call>
{"name": <function-name>, "arguments": <args-dict>, "id": <monotonically-increasing-id>}
</tool_call>

- Do NOT make assumptions about parameter values.
- Pay close attention to each parameter's `type` and use the correct Python types.
- Use exactly one tool call per turn, and increment `id` for each call.

## Available Tools
<tools>
%s
</tools>

## Example Session
<question>What's the current temperature in Madrid?</question>
<thought>I need to retrieve the current weather in Madrid.</thought>
<tool_call>{"name": "get_current_weather", "arguments": {"location": "Madrid", "unit": "celsius"}, "id": 0}</tool_call>
<observation>{0: {"temperature": 25, "unit": "celsius"}}</observation>
<thought>I now have the temperature for Madrid. I can respond to the user.</thought>
<response>The current temperature in Madrid is 25 degrees Celsius.</response>

## Additional Constraints
- Do not skip directly to a final answer while important evidence is still missing.
- Keep the final answer evidence-bounded: state only what the observations support.
- Never leave the loop without eventually producing a <response>.
"""


class ReactAgent:
    """Reason -> Action -> Observation loop. The model thinks, calls one tool, reads the result,
    and repeats until it emits a <response>."""

    def __init__(self, tools, model="openai/gpt-oss-120b", client=None, system_prompt=""):
        if client is None:
            raise ValueError("ReactAgent requires a `client` (chat.completions.create).")
        self.client = client
        self.model = model
        self.system_prompt = system_prompt
        self.tools = tools if isinstance(tools, list) else [tools]
        self.tools_dict = {t.name: t for t in self.tools}

    def add_tool_signatures(self) -> str:
        return "".join(t.fn_signature for t in self.tools)

    def process_tool_calls(self, tool_calls_content: list) -> dict:
        observations = {}
        for tool_call_str in tool_calls_content:
            try:
                tool_call = json.loads(tool_call_str)
            except json.JSONDecodeError:
                print(f"\nSkipping malformed tool call: {tool_call_str}")
                continue
            tool_name = tool_call.get("name") if isinstance(tool_call, dict) else None
            t = self.tools_dict.get(tool_name)
            if t is None:
                print(f"\nSkipping unknown tool request: {tool_name}")
                observations[len(observations)] = f"Unknown tool {tool_name!r}. Use one of: {', '.join(self.tools_dict)}."
                continue
            print(f"\nUsing Tool: {tool_name}")
            tool_call.setdefault("arguments", {})
            try:
                validated = validate_arguments(tool_call, json.loads(t.fn_signature))
                print(f"Tool call: {validated}")
                result = t.run(**validated["arguments"])
            except Exception as e:   # a wrong or missing argument becomes an observation, not a crash
                validated, result = tool_call, f"Tool error: {e}"
            print(f"Tool result: {result}")
            observations[validated.get("id", len(observations))] = result
        return observations

    def run(self, user_msg: str, max_rounds: int = 10) -> str:
        system_prompt = (self.system_prompt + "\n" + REACT_SYSTEM_PROMPT % self.add_tool_signatures()).strip()
        chat_history = ChatHistory([
            build_prompt_structure(system_prompt, "system"),
            build_prompt_structure(user_msg, "user", tag="question"),
        ])
        tools_used, pushbacks = 0, 0
        for _ in range(max_rounds):
            completion = completions_create(self.client, chat_history, self.model)

            thought = extract_tag_content(str(completion), "thought")
            tool_calls = extract_tag_content(str(completion), "tool_call")
            response = extract_tag_content(str(completion), "response")
            if not str(completion).strip():       # some models plan the call silently and send nothing
                update_chat_history(chat_history, "Your reply was empty. Write your <thought> and "
                                    "<tool_call> tags as plain text in the reply.", "user")
                continue
            # A <response> only counts when no tool call came with it. A model that writes a tool call,
            # imagines its result, and answers in one go must run the tool and see the REAL result first.
            if response.found and not tool_calls.found:
                if self.tools_dict and tools_used == 0 and pushbacks < 2:
                    # Evidence check: an answer before any tool has run cannot be based on the evidence.
                    pushbacks += 1
                    update_chat_history(chat_history, completion, "assistant")
                    update_chat_history(chat_history, "You have not called any tool yet, so that answer is "
                                        "not based on evidence. Call a tool first with a <tool_call> block.", "user")
                    continue
                return response.content[0]
            if tool_calls.found:
                completion = str(completion).split("<observation>")[0].split("<response>")[0]
            update_chat_history(chat_history, completion, "assistant")

            if thought.found and thought.content:
                print(f"\nThought: {thought.content[0]}")

            if tool_calls.found:
                observations = self.process_tool_calls(tool_calls.content)
                tools_used += len(observations)
                print(f"\nObservations: {observations}")
                update_chat_history(chat_history, f"<observation>{observations}</observation>", "user")

        return completions_create(self.client, chat_history, self.model)


# ---------------------------------------------------------------- multiagent: Agent + Crew
from textwrap import dedent


class Agent:
    """A team member: a ReactAgent with a role (backstory), a task, and dependencies.

    Dependencies are declared with the `>>` operator (a >> b means b depends on a, and a's output
    becomes b's context). Adapted from The Neural Maze's multiagent_pattern/agent.py.
    """

    def __init__(self, name, backstory, task_description, task_expected_output="",
                 tools=None, llm="openai/gpt-oss-120b", client=None):
        self.name = name
        self.backstory = backstory
        self.task_description = task_description
        self.task_expected_output = task_expected_output
        self.react_agent = ReactAgent(model=llm, tools=tools or [], client=client)
        self.react_agent.system_prompt = backstory  # role instructions
        self.dependencies = []
        self.dependents = []
        self.context = ""
        Crew.register_agent(self)

    def __repr__(self):
        return self.name

    def __rshift__(self, other):
        self.add_dependent(other)
        return other

    def add_dependency(self, other):
        if isinstance(other, Agent):
            self.dependencies.append(other); other.dependents.append(self)
        elif isinstance(other, list):
            for item in other:
                self.dependencies.append(item); item.dependents.append(self)
        else:
            raise TypeError("dependency must be an Agent or list of Agents")

    def add_dependent(self, other):
        if isinstance(other, Agent):
            other.dependencies.append(self); self.dependents.append(other)
        elif isinstance(other, list):
            for item in other:
                item.dependencies.append(self); self.dependents.append(item)
        else:
            raise TypeError("dependent must be an Agent or list of Agents")

    def receive_context(self, input_data):
        self.context += f"\n{self.name} received context:\n{input_data}\n"

    def create_prompt(self):
        return dedent(f"""
        You are an AI agent on a team working together to complete a forensic task.
        The task is in <task_description></task_description>. Context from earlier agents is in
        <context></context> (empty if none). The required output format is in
        <task_expected_output></task_expected_output>. Produce the best response, following that format.

        <task_description>
        {self.task_description}
        </task_description>

        <task_expected_output>
        {self.task_expected_output}
        </task_expected_output>

        <context>
        {self.context}
        </context>

        Your response:
        """).strip()

    def run(self):
        output = self.react_agent.run(user_msg=self.create_prompt())
        for dependent in self.dependents:
            dependent.receive_context(output)
        return output


class Crew:
    """Holds a team of agents and runs them in dependency (topological) order.

    Adapted from The Neural Maze's multiagent_pattern/crew.py (graphviz plotting omitted; use
    `describe()` for a text view of the dependency graph).
    """

    current_crew = None

    def __init__(self):
        self.agents = []

    def __enter__(self):
        Crew.current_crew = self
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        Crew.current_crew = None

    def add_agent(self, agent):
        self.agents.append(agent)

    @staticmethod
    def register_agent(agent):
        if Crew.current_crew is not None:
            Crew.current_crew.add_agent(agent)

    def topological_sort(self):
        from collections import deque
        in_degree = {a: len(a.dependencies) for a in self.agents}
        queue = deque([a for a in self.agents if in_degree[a] == 0])
        ordered = []
        while queue:
            a = queue.popleft()
            ordered.append(a)
            for dep in a.dependents:
                in_degree[dep] -= 1
                if in_degree[dep] == 0:
                    queue.append(dep)
        if len(ordered) != len(self.agents):
            raise ValueError("Circular dependency detected among agents")
        return ordered

    def describe(self):
        lines = ["Dependency graph (who feeds whom):"]
        for a in self.agents:
            deps = ", ".join(d.name for d in a.dependencies) or "(none)"
            lines.append(f"  {a.name}  <- depends on: {deps}")
        lines.append("Run order: " + " -> ".join(a.name for a in self.topological_sort()))
        return "\n".join(lines)

    def run(self):
        outputs = {}
        for agent in self.topological_sort():
            print(f"\n{'=' * 50}\nRUNNING AGENT: {agent.name}\n{'=' * 50}")
            out = agent.run()
            outputs[agent.name] = out
            print(out)
        return outputs

## The Case in One Minute

**Case `MA-2026-022`** - a public-health outreach phone was left unattended during a vaccination
event. The question: was `patients_contacts.png` transmitted from the phone, how confident can we be,
and does a gap in the chain-of-custody record weaken that confidence?

| UTC time | What the records show |
|---|---|
| 18:35:02 | device locked (start of the unattended interval) |
| 18:44:11 | `patients_contacts.png` created on the device |
| 18:45:02 | SecureChat attach attempt with that file |
| 18:45:07 | network `upload_started` (48,211 bytes out) |
| 19:10:08 | device unlocked |
| (later) | an analyst review begins with **no documented transfer** from the prior handler |

The records show an upload **started**, not an upload **completed**, so transmission is at most
**likely**, not confirmed. And the custody gap is an **evidence-handling** issue, not proof about the
phone. Keeping those two questions separate is the heart of the lab.

## How a Multiagent Team Works

Three specialists, each with a narrow job, feeding the next:

| Agent | Role | Hands forward |
|---|---|---|
| **InvestigationAgent** | Reconstruct the technical timeline. | A draft transmission assessment. |
| **EvidenceVerificationAgent** | Test those claims against the records. | A revised, evidence-bounded rating. |
| **CustodyAuditAgent** | Review evidence handling; write the conclusion. | The final confidence-bounded answer. |

Each agent is itself a ReAct agent with tools (Lab 03), so it reasons and calls evidence tools inside
its own turn. What is new is the **handoff**: a later agent automatically receives the earlier agents'
outputs as context, and is told to **challenge and narrow** them, not just agree.

## Part A: Run the Team One Agent at a Time

You will define the evidence tools, build the three agents with their dependencies, and run them in
order so you can see each contribution and how context moves across the team.

### Step A.1: Define the evidence tools

**Purpose:** wrap each artifact type in a small tool. **Technical** tools read device, file, messaging,
and network records; the **custody** tool reads the handling log. Splitting them keeps the technical
transmission question separate from the custody question.

In [ ]:
# Purpose: define the technical and custody evidence tools.
import csv   # csv: reads spreadsheet-style files (rows of comma-separated values)
import json  # json: reads and writes structured data (lists and labelled fields)
from pathlib import Path
from IPython.display import Markdown, display  # show nicely formatted text (headings, bold) in the notebook

data_dir = Path('data')

def read_csv_rows(filename):
    with (data_dir / filename).open(newline='') as h:
        return list(csv.DictReader(h))

@tool
def list_artifact_files():
    """Return the available artifact files for Lab 05."""
    return sorted(p.name for p in data_dir.iterdir() if p.is_file())

@tool
def get_device_state_events():
    """Return the lock and unlock records that bound the unattended interval."""
    return read_csv_rows('device_state.csv')

@tool
def get_patients_file_event():
    """Return the file event for patients_contacts.png."""
    return read_csv_rows('file_events.csv')

@tool
def get_securechat_attachment_event():
    """Return the SecureChat attachment-attempt record for patients_contacts.png."""
    return read_csv_rows('messaging_events.csv')

@tool
def get_securechat_network_event():
    """Return the SecureChat network upload record."""
    return read_csv_rows('network_events.csv')

@tool
def get_chain_of_custody_events():
    """Return the chain-of-custody records used to judge evidence-handling completeness."""
    return read_csv_rows('chain_of_custody.csv')

technical_tools = [list_artifact_files, get_device_state_events, get_patients_file_event,
                   get_securechat_attachment_event, get_securechat_network_event]
custody_tools = [list_artifact_files, get_chain_of_custody_events]
print('technical tools:', [t.name for t in technical_tools])
print('custody tools:  ', [t.name for t in custody_tools])

### Step A.2: Build the three agents and their dependencies

**Purpose:** give each agent a role (backstory), a task, an output format, and its tools, then wire the
handoffs with `>>`. `InvestigationAgent >> EvidenceVerificationAgent` means verification depends on
investigation and receives its output as context.  
**Output:** three connected agents. Custody depends on **both** earlier agents.

In [ ]:
# Purpose: define the shared case context and a helper that builds the wired team.
LAB_QUESTION = (
    'Was patients_contacts.png transmitted from the outreach phone, should the conclusion be labeled '
    'confirmed, likely, or unconfirmed, and does the missing transfer in the chain-of-custody log '
    'weaken confidence in that conclusion?')

CASE_SCOPE = '''Case facts:
- Incident window: 2026-03-01T18:35:00Z to 2026-03-01T19:10:00Z.
- File of interest: patients_contacts.png.
- Technical artifacts: device-state, file, messaging, and network records.
- Evidence-handling review uses chain_of_custody.csv.
- The missing transfer is an undocumented custody handoff between analysts, NOT proof about whether the phone completed transmission.
- The final conclusion must use one label: confirmed, likely, or unconfirmed.'''

def build_forensic_agents():
    investigation = Agent(
        name='InvestigationAgent',
        backstory='You reconstruct mobile-forensics timelines. Order events carefully and do not overclaim upload completion.',
        task_description=f'{LAB_QUESTION}\n\n{CASE_SCOPE}\n\nUse the technical evidence tools to reconstruct the timeline and give an initial transmission assessment.',
        task_expected_output='Use these labels:\nInvestigation timeline:\nInitial transmission assessment:\nOpen technical questions:',
        tools=technical_tools, llm=MODEL, client=client)
    verification = Agent(
        name='EvidenceVerificationAgent',
        backstory='You verify whether technical claims are supported by specific artifact records, and keep ratings evidence-bounded.',
        task_description=f'{LAB_QUESTION}\n\n{CASE_SCOPE}\n\nUse the technical tools to test the earlier claims. If the artifacts show only file creation, an attach attempt, and upload_started, say so. Do not treat upload_started as confirmed completion.',
        task_expected_output='Use these labels:\nClaims checked:\nSupported evidence:\nUnsupported or missing evidence:\nRevised transmission rating:',
        tools=technical_tools, llm=MODEL, client=client)
    custody = Agent(
        name='CustodyAuditAgent',
        backstory='You review chain-of-custody records and explain how handling gaps affect final confidence, using earlier technical findings as context.',
        task_description=f'{LAB_QUESTION}\n\n{CASE_SCOPE}\n\nUse the chain-of-custody tool and the earlier context to write the final conclusion. Check for a missing handoff between handlers. Treat any custody gap as an evidence-handling issue, not proof about the phone. Explain how it affects the final label.',
        task_expected_output='Use these labels:\nVerified technical finding:\nCustody review:\nConfidence impact:\nFinal conclusion:',
        tools=custody_tools, llm=MODEL, client=client)
    investigation >> verification
    investigation >> custody
    verification >> custody
    return investigation, verification, custody

investigation_agent, evidence_verification_agent, custody_audit_agent = build_forensic_agents()
print('Custody depends on:', [a.name for a in custody_audit_agent.dependencies])

### Step A.3: Run InvestigationAgent

**Purpose:** the first specialist builds the technical timeline and a draft assessment. It should treat
file creation, the attach attempt, and `upload_started` as an **attempt**, not confirmed completion.

In [ ]:
investigation_output = investigation_agent.run()
display(Markdown('### InvestigationAgent Output\n\n' + investigation_output))

### Step A.4: Run EvidenceVerificationAgent

**Purpose:** the verifier receives the investigation output as context (shown first), then tests it
against the records. It should lower any claim that treats `upload_started` as a completed transfer.

In [ ]:
display(Markdown('### EvidenceVerificationAgent received this context\n\n```text\n' + evidence_verification_agent.context.strip() + '\n```'))
evidence_verification_output = evidence_verification_agent.run()
display(Markdown('### EvidenceVerificationAgent Output\n\n' + evidence_verification_output))

### Step A.5: Run CustodyAuditAgent

**Purpose:** the auditor receives **both** earlier outputs, then checks the handling record and writes
the final conclusion. It must keep the custody gap separate from the transmission question.

In [ ]:
display(Markdown('### CustodyAuditAgent received this context\n\n```text\n' + custody_audit_agent.context.strip() + '\n```'))
custody_audit_output = custody_audit_agent.run()
display(Markdown('### CustodyAuditAgent Final Conclusion\n\n' + custody_audit_output))

## Part B: Orchestrate the Same Team with `Crew`

`Crew` holds the dependency graph in one place and runs the agents in the right order for you. The case
logic is identical; only the orchestration changes.

### Step B.1: Build the team inside a Crew and show the graph

**Purpose:** rebuild a fresh team inside a `Crew` context (which registers the agents), then print the
dependency graph and the run order it computes.

In [ ]:
with Crew() as crew:
    build_forensic_agents()

print(crew.describe())

### Step B.2: Run the full crew

**Purpose:** `Crew` runs each agent in dependency order and passes earlier outputs forward. Compare the
final conclusion with your Part A run. Wording varies by run; what should not change is that the
conclusion stays evidence-bounded.

In [ ]:
crew_outputs = crew.run()
display(Markdown('### Crew Final Conclusion (CustodyAuditAgent)\n\n' + crew_outputs['CustodyAuditAgent']))

## Your Final Judgment

The agents are aids, not the authority. Write your own conclusion below, based on the records and the
agent outputs.

1. **Transmission label:** confirmed / likely / unconfirmed, and why.
2. **Technical basis:** which records support it (and the `upload_started` vs `upload_completed` point).
3. **Custody impact:** how the missing handoff affects confidence, kept separate from the phone question.
4. **Remaining uncertainty:** what the evidence does not establish.

In [ ]:
final = {
    'transmission_label': '... confirmed / likely / unsupported ...',
    'technical_basis': '... the timestamps and records behind your label ...',
    'custody_impact': '... what the chain-of-custody gap does and does not affect ...',
    'uncertainty': '... what no record confirms ...',
}
for k, v in final.items():
    print(k, '->', v)

### What to notice

1. Did `InvestigationAgent` keep the timeline clear without overstating completion?
2. Did `EvidenceVerificationAgent` lower confidence on seeing only `upload_started`?
3. Did `CustodyAuditAgent` explain the custody gap without turning it into proof about the phone?
4. Did you keep the transmission question separate from the evidence-handling question?

## Submit

Save the notebook with Part A, Part B, and your judgment filled in.

- In Colab: **File -> Download -> Download .ipynb**, or **File -> Save a copy in Drive**.

---

*Case data and the lab concept are adapted from*
*[frankwxu/agentic-AI4-forensics](https://github.com/frankwxu/agentic-AI4-forensics). The multiagent
framework is adapted from [The Neural Maze's agentic-patterns-course](https://github.com/neural-maze/agentic-patterns-course) (MIT License).*